# AHR_RL: train a pre-race greening agent on the full Betfair recordings

This notebook runs the `ahr_rl` package end-to-end on **all** recordings in
`My Drive/betfair stream data/recordings`:

1. compile each `*.ndjson.gz` into a compact tape (cached back to Drive)
2. sanity baselines (do-nothing / random / rule-based scalper)
3. optional: planted-edge sanity check on synthetic races
4. PPO training (GPU if available)
5. held-out TEST evaluation vs baselines

Runtime: GPU (T4 is fine). The env simulation is CPU-bound, so more CPU cores help more than a bigger GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/betfair stream data'

In [ ]:
!git clone -b claude/wizardly-clarke-rb37hl https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL || (cd /content/AHR_RL && git pull)
%cd /content/AHR_RL
!pip -q install orjson gymnasium>=1.0
!python -m pytest -q tests/test_exchange.py

## 1. Compile recordings -> tapes (incremental; cached on Drive)

In [ ]:
import os
TAPES = f'{DRIVE}/rl_tapes_v1'
os.makedirs(TAPES, exist_ok=True)
!python -m ahr_rl.tape "{DRIVE}/recordings" "{TAPES}" --workers {os.cpu_count()}
# local copy is much faster to read than the Drive mount
!mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!ls /content/tapes | wc -l

## 2. Baselines on the most recent 150 races

In [ ]:
import glob
from ahr_rl.evaluate import evaluate, summarise, noop_policy, make_random_policy, make_scalper_policy
tapes = sorted(glob.glob('/content/tapes/*.npz'))
recent = tapes[-150:]
for name, pol in [('do_nothing', noop_policy), ('random_1pct', make_random_policy(0.01)), ('scalper_rule', make_scalper_policy())]:
    print(name, {k: round(v, 2) for k, v in summarise(evaluate(pol, recent)).items()})

## 3. (optional) Planted-edge sanity check
The agent should learn to profit on synthetic races where one runner steadily shortens. If it can't, fix the learner before blaming the market.

In [ ]:
RUN_SYNTHETIC = False
if RUN_SYNTHETIC:
    !python -m ahr_rl.synthetic /content/synthetic 200
    !python -m ahr_rl.train --tapes "/content/synthetic/*.npz" --out /content/runs/synthetic --total-steps 500000 --n-envs {os.cpu_count()} --eval-every 25

## 4. Train on the real recordings
Chronological split: oldest ~70% of race days train, next ~15% validation (checkpoint selection), newest ~15% test (untouched until step 5).

In [ ]:
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
!python -m ahr_rl.train --tapes "/content/tapes/*.npz" --out "{RUN}" --total-steps 5000000 --n-envs {max(4, os.cpu_count())} --n-steps 256 --eval-every 25 --eval-races 200

In [ ]:
import pandas as pd
log = pd.read_csv(f'{RUN}/train_log.csv')
ax = log.plot(x='steps', y=['ep_green_mean'], figsize=(10, 4), title='train: mean worst-case $ per race (last 50)')
log.dropna(subset=['val_green']).plot(x='steps', y=['val_green'], ax=ax, marker='o')

## 5. Held-out test evaluation
`mean_green_$` = average guaranteed (worst-case, after commission) profit per race. `do_nothing` scores exactly 0, so the agent has to beat 0 on unseen races to be worth running.

In [ ]:
!python -m ahr_rl.report --run "{RUN}" --split test

## 6. Paper-trade live
On a machine with Betfair API credentials (not Colab):
```
export BF_USERNAME=... BF_PASSWORD=... BF_APP_KEY=... BF_CERTS=/path/to/certs
python -m ahr_rl.live.bot --model runs/ppo_v1/best.pt          # paper (default)
```